# 05 Hyperparametric Tuning

This notebook performs hyperparameter optimization using **Optuna** to find the best learning rate, dropout, and batch size for the image classification model.

In [5]:
import json
from pathlib import Path

import optuna
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image, UnidentifiedImageError
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import models, transforms

DATA_ROOT = Path("../../data/raw/images")
SPLITS = ["train", "val", "test"]
CLASSES = ["real", "fake"]
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## 1. Data Collection and Dataset Logic

In [6]:
def collect_image_paths(data_root: Path) -> pd.DataFrame:
    rows = []
    for split in SPLITS:
        for class_name in CLASSES:
            class_dir = data_root / split / class_name
            if not class_dir.exists():
                continue
            for path in sorted(class_dir.iterdir()):
                if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS:
                    rows.append({
                        "path": str(path), 
                        "split": split, 
                        "label": class_name, 
                        "label_id": 1 if class_name == "fake" else 0
                    })
    return pd.DataFrame(rows)

class DeepfakeImageDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def _load_image(self, path: str):
        try:
            with Image.open(path) as image:
                return image.convert("RGB")
        except (OSError, UnidentifiedImageError):
            return Image.new("RGB", (224, 224), color=(0, 0, 0))

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        image = self._load_image(row["path"])
        label = torch.tensor(row["label_id"], dtype=torch.float32)
        if self.transform is not None:
            image = self.transform(image)
        return image, label

df = collect_image_paths(DATA_ROOT)
print(f"Total images found: {len(df)}")

Total images found: 190335


## 2. Create Balanced Subset (CRITICAL)

In [7]:
# Filter training split
df_train = df[df.split == "train"]

# Create balanced subset of max 10,000 samples
df_real = df_train[df_train.label_id == 0].sample(min(5000, len(df_train[df_train.label_id == 0])), random_state=42)
df_fake = df_train[df_train.label_id == 1].sample(min(5000, len(df_train[df_train.label_id == 1])), random_state=42)
df_subset = pd.concat([df_real, df_fake]).sample(frac=1, random_state=42)

# Safety Check
print("Dataset size:", len(df_subset))
print("Class distribution:\n", df_subset.label.value_counts())

Dataset size: 10000
Class distribution:
 label
fake    5000
real    5000
Name: count, dtype: int64


## 3. Training Utilities

In [8]:
def build_model(dropout: float) -> nn.Module:
    weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1
    model = models.efficientnet_b0(weights=weights)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=dropout, inplace=True),
        nn.Linear(in_features, 1)
    )
    return model

def train_for_few_epochs(model, train_loader, val_loader, optimizer, epochs=1):
    criterion = nn.BCEWithLogitsLoss()
    
    for epoch in range(epochs):
        # Train
        model.train()
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device).unsqueeze(1)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()
            
        # Validation
        model.eval()
        val_loss = 0.0
        total = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device).unsqueeze(1)
                loss = criterion(model(images), labels)
                val_loss += loss.item() * images.size(0)
                total += images.size(0)
        
        final_val_loss = val_loss / max(total, 1)
        
    return None, final_val_loss

## 4. Optuna Objective

In [9]:
def objective(trial):
    # Hyperparameters
    lr = trial.suggest_float("lr", 1e-5, 1e-3, log=True)
    dropout = trial.suggest_float("dropout", 0.2, 0.5)
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])

    # Model
    model = build_model(dropout).to(device)

    # Data Loaders
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    
    # Split the subset into train/val for the trial
    df_trial_train = df_subset.sample(frac=0.8, random_state=42)
    df_trial_val = df_subset.drop(df_trial_train.index)
    
    train_ds = DeepfakeImageDataset(df_trial_train, transform=transform)
    val_ds = DeepfakeImageDataset(df_trial_val, transform=transform)
    
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, pin_memory=True)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    # Short training
    _, val_loss = train_for_few_epochs(model, train_loader, val_loader, optimizer, epochs=1)

    return val_loss

## 5. Run Study

In [10]:
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=20)

[I 2026-05-10 08:58:09,071] A new study created in memory with name: no-name-85993b4a-20f2-403b-b4a2-117f2eeff23b
[I 2026-05-10 08:59:27,535] Trial 0 finished with value: 0.09774650990217924 and parameters: {'lr': 0.00013519336175097903, 'dropout': 0.3431004603181488, 'batch_size': 16}. Best is trial 0 with value: 0.09774650990217924.
[I 2026-05-10 09:04:32,994] Trial 1 finished with value: 0.07984104898199439 and parameters: {'lr': 0.0005508671788191697, 'dropout': 0.2851019686282458, 'batch_size': 64}. Best is trial 1 with value: 0.07984104898199439.
[I 2026-05-10 09:09:32,790] Trial 2 finished with value: 0.10938993561267853 and parameters: {'lr': 0.00010496299635857632, 'dropout': 0.44614242346098754, 'batch_size': 64}. Best is trial 1 with value: 0.07984104898199439.
[I 2026-05-10 09:13:29,410] Trial 3 finished with value: 0.09220212783943861 and parameters: {'lr': 0.0007961235023415899, 'dropout': 0.4119908739037844, 'batch_size': 16}. Best is trial 1 with value: 0.07984104898199

## 6. Results

In [11]:
print("Best Params:", study.best_params)

with open("best_params.json", "w") as f:
    json.dump(study.best_params, f)

print("Best parameters saved to best_params.json")

Best Params: {'lr': 0.0004029469429975904, 'dropout': 0.4955353561591949, 'batch_size': 32}
Best parameters saved to best_params.json
